In [4]:
mkdir "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/tumor_detection_and_classification/ResNet50"

In [4]:
# import torch
# import torch.nn as nn
# import torchvision.models as models
# import torch.optim as optim
# from torch.utils.data import DataLoader, Dataset
# import numpy as np

# # CNN Backbone - Pretrained ResNet50 (you can replace it with DenseNet or another model)
# class CNNBackbone(nn.Module):
#     def __init__(self, pretrained=True):
#         super(CNNBackbone, self).__init__()
#         self.resnet = models.resnet50(pretrained=pretrained)
#         self.resnet.fc = nn.Identity()  # Removing the final classifier layer

#     def forward(self, x):
#         return self.resnet(x)  # Returns feature maps

# # Transformer Encoder
# class TransformerEncoder(nn.Module):
#     def __init__(self, input_dim, num_heads=8, num_layers=6):
#         super(TransformerEncoder, self).__init__()
#         self.encoder_layer = nn.TransformerEncoderLayer(d_model=input_dim, nhead=num_heads)
#         self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)

#     def forward(self, x):
#         return self.transformer_encoder(x)

# # Multi-Task Learning Model
# class MultiTaskModel(nn.Module):
#     def __init__(self, cnn_backbone, transformer_encoder, num_classes=2):
#         super(MultiTaskModel, self).__init__()
        
#         self.cnn_backbone = cnn_backbone
#         self.transformer_encoder = transformer_encoder
        
#         # Classification head
#         self.classification_head = nn.Sequential(
#             nn.Linear(2048, 512),  # Assuming CNN backbone output of 2048 channels
#             nn.ReLU(),
#             nn.Linear(512, num_classes)
#         )
        
#         # Segmentation head
#         self.segmentation_head = nn.Sequential(
#             nn.ConvTranspose2d(2048, 512, kernel_size=4, stride=2, padding=1),
#             nn.ReLU(),
#             nn.ConvTranspose2d(512, 1, kernel_size=4, stride=2, padding=1),
#             nn.Sigmoid()  # For binary mask output
#         )

#     def forward(self, x):
#         # Pass through CNN backbone
#         features = self.cnn_backbone(x)
        
#         # Flatten or reshape features for transformer input (if necessary)
#         features = features.view(features.size(0), -1)  # Flatten for transformer
        
#         # Pass through Transformer Encoder
#         transformer_out = self.transformer_encoder(features)
        
#         # Classification output
#         classification_out = self.classification_head(transformer_out)
        
#         # Reshape features for segmentation task (e.g., reshaping to original image size)
#         segmentation_out = self.segmentation_head(features)
        
#         return classification_out, segmentation_out

# # Loss function combining classification and segmentation loss
# classification_loss = nn.CrossEntropyLoss()  # For tumor classification
# segmentation_loss = nn.BCELoss()  # For segmentation mask (binary mask)

# def multi_task_loss(classification_out, classification_target, segmentation_out, segmentation_target):
#     # Classification loss
#     class_loss = classification_loss(classification_out, classification_target)
    
#     # Segmentation loss
#     seg_loss = segmentation_loss(segmentation_out, segmentation_target)
    
#     # Combine the losses with weighting (e.g., 0.5 for each task)
#     total_loss = 0.5 * class_loss + 0.5 * seg_loss
#     return total_loss

# # Example Dataset for illustration purposes (replace with your actual dataset)
# class TumorDataset(Dataset):
#     def __init__(self, images, class_labels, seg_masks, transform=None):
#         self.images = images
#         self.class_labels = class_labels
#         self.seg_masks = seg_masks
#         self.transform = transform

#     def __len__(self):
#         return len(self.images)

#     def __getitem__(self, idx):
#         image = self.images[idx]
#         class_label = self.class_labels[idx]
#         seg_mask = self.seg_masks[idx]
        
#         if self.transform:
#             image = self.transform(image)  # Apply any transformations (e.g., normalization)
        
#         return image, class_label, seg_mask

# # Initialize the model
# cnn_backbone = CNNBackbone()
# transformer_encoder = TransformerEncoder(input_dim=2048)  # Adjust input_dim based on CNN output
# model = MultiTaskModel(cnn_backbone=cnn_backbone, transformer_encoder=transformer_encoder)

# # Optimizer
# optimizer = optim.Adam(model.parameters(), lr=1e-4)

# # Example of data loading (replace with actual data)
# # Assuming `images`, `class_labels`, and `seg_masks` are lists or numpy arrays
# # images: List of images, class_labels: Tumor classification labels, seg_masks: Tumor segmentation masks
# images = np.random.randn(100, 3, 224, 224)  # Random image data (replace with actual)
# class_labels = np.random.randint(0, 2, size=(100,))  # Random binary labels (0/1)
# seg_masks = np.random.randn(100, 1, 224, 224)  # Random binary masks (replace with actual)

# # DataLoader
# dataset = TumorDataset(images, class_labels, seg_masks)
# dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# # Training Loop
# num_epochs = 10
# for epoch in range(num_epochs):
#     model.train()
#     running_loss = 0.0
#     for data in dataloader:
#         images, class_labels, seg_masks = data
        
#         # Convert data to PyTorch tensors
#         images = torch.tensor(images, dtype=torch.float32)
#         class_labels = torch.tensor(class_labels, dtype=torch.long)
#         seg_masks = torch.tensor(seg_masks, dtype=torch.float32)
        
#         # Zero gradients
#         optimizer.zero_grad()
        
#         # Forward pass
#         class_out, seg_out = model(images)
        
#         # Calculate loss
#         loss = multi_task_loss(class_out, class_labels, seg_out, seg_masks)
        
#         # Backward pass and optimization
#         loss.backward()
#         optimizer.step()
        
#         running_loss += loss.item()

#     print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(dataloader)}")


In [ ]:
ls "/mnt/Internal/MedImage/Datasets/Kits23/8//"

#### KITS23 - Dataset

In [ ]:
# Kits23 Multi-Task Tumor Detection and Segmentation
import os
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from PIL import Image

# Configuration parameters
IMAGE_ROOT = "/mnt/Internal/MedImage/Datasets/Kits23/8/AUGMENTED/DATASET_FINAL/JPEGImages"
ANNOTATION_ROOT = "/mnt/Internal/MedImage/Datasets/Kits23/8/AUGMENTED/DATASET_FINAL/Annotations"
model_dir = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/tumor_detection_and_classification/ResNet50"
START_EPOCH = 0
EPOCHS = 5
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
SAVE_EVERY = 1

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Save Checkpoint Function
def save_checkpoint(model, optimizer, epoch, save_path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, save_path)

# Inside visualize_predictions: update it to handle batches correctly
def visualize_predictions(predictions, ground_truths, case_ids):
    plt.figure(figsize=(12, 4))
    for i in range(min(3, len(predictions))):  # Show up to 3
        pred = predictions[i][0] if predictions[i].shape[0] == 1 else predictions[i][0, 0]
        gt = ground_truths[i][0] if ground_truths[i].shape[0] == 1 else ground_truths[i][0, 0]

        plt.subplot(2, 3, i + 1)
        plt.imshow(pred, cmap='gray')
        plt.title(f"Pred Case {case_ids[i]}")
        plt.axis('off')

        plt.subplot(2, 3, i + 4)
        plt.imshow(gt, cmap='gray')
        plt.title(f"GT Case {case_ids[i]}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

class Kits23Dataset(Dataset):
    def __init__(self, image_dir, annotation_dir, labels, transform=None):
        self.image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith('.jpg') or f.endswith('.jpeg')]
        self.mask_paths = [os.path.join(annotation_dir, f.replace('.jpg', '_segmentation.png').replace('.jpeg', '_segmentation.png')) for f in os.listdir(image_dir)]
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = Image.open(image_path).convert('RGB')

        if not os.path.exists(mask_path):
            mask_path = mask_path.replace('.jpeg', '.jpg')
        if not os.path.exists(mask_path):
            print(f"Warning: Mask not found for {image_path}, using a dummy mask.")
            mask = Image.fromarray(np.zeros((256, 256), dtype=np.uint8))
        else:
            mask = Image.open(mask_path).convert('L')

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
        else:
            image = transforms.ToTensor()(image)
            mask = transforms.ToTensor()(mask)

        label = self.labels[idx]
        return image, torch.tensor(label, dtype=torch.long), mask

    
from sklearn.model_selection import train_test_split

# Dummy labels for testing (replace with actual labels later)
labels = [0] * len(os.listdir(IMAGE_ROOT))

# Dataset Initialization
dataset = Kits23Dataset(
    image_dir=IMAGE_ROOT,
    annotation_dir=ANNOTATION_ROOT,
    labels=labels
)

# Train-validation split using sklearn
indices = list(range(len(dataset)))
train_indices, val_indices = train_test_split(indices, test_size=0.2, random_state=42)

# Subset and create DataLoaders
train_dataset = torch.utils.data.Subset(dataset, train_indices)
val_dataset = torch.utils.data.Subset(dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# Print dataset info
print(f"Total samples: {len(dataset)}")
print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")


import torch
import torch.nn as nn
from torchvision import models

# CNN Backbone - Pretrained ResNet50
class CNNBackbone(nn.Module):
    def __init__(self, pretrained=True):
        super(CNNBackbone, self).__init__()
        base_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        # Extract convolutional feature maps (remove avgpool and fc layers)
        self.features = nn.Sequential(*list(base_model.children())[:-2])  # Output: (B, 2048, H/32, W/32)

    def forward(self, x):
        x = self.features(x)  # Shape: (B, 2048, H/32, W/32)
        return x

# Transformer Encoder with reshaped CNN features
class TransformerEncoder(nn.Module):
    def __init__(self, input_dim=2048, num_heads=8, num_layers=6):
        super(TransformerEncoder, self).__init__()
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=input_dim, nhead=num_heads, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=num_layers)

    def forward(self, x):
        # x is expected to be [B, Seq_len, C] already
        return self.transformer_encoder(x)  # Output: [B, Seq_len, C]

    

import torch
import torch.nn as nn

class MultiTaskModel(nn.Module):
    def __init__(self, cnn_backbone, transformer_encoder, num_classes=2):
        super(MultiTaskModel, self).__init__()
        self.cnn_backbone = cnn_backbone
        self.transformer_encoder = transformer_encoder

        self.classification_head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

        self.segmentation_head = nn.Sequential(
            nn.ConvTranspose2d(2048, 1024, 4, 2, 1),  # 1x1 -> 2x2
            nn.ReLU(),
            nn.ConvTranspose2d(1024, 512, 4, 2, 1),   # 2x2 -> 4x4
            nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1),    # 4x4 -> 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),    # 8x8 -> 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),     # 16x16 -> 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(64, 1, 4, 2, 1),       # 32x32 -> 64x64
            nn.Sigmoid()
        )

    def forward(self, x):
        cnn_features = self.cnn_backbone(x)        # [B, 2048, 1, 1]
        flat_features = cnn_features.view(x.size(0), -1)  # [B, 2048]

        # Transformer for classification only
        features_seq = flat_features.unsqueeze(1)
        features_seq = features_seq.permute(1, 0, 2)         # [B, 1, 2048]
        transformer_out = self.transformer_encoder(features_seq)  # [B, 1, 2048]
        transformer_out = transformer_out.permute(1, 0, 2).squeeze(1)      # [B, 2048]
        classification_out = self.classification_head(transformer_out)

        # Segmentation uses CNN feature maps directly
        segmentation_out = self.segmentation_head(cnn_features)

        return classification_out, segmentation_out



# Loss Function
classification_loss = nn.CrossEntropyLoss()
segmentation_loss = nn.BCELoss()


# Combined Multi-Task Loss
def multi_task_loss(class_out, class_target, seg_out, seg_target):
    class_loss = classification_loss(class_out, class_target)
    seg_loss = segmentation_loss(seg_out, seg_target)
    return 0.5 * class_loss + 0.5 * seg_loss


# Initialize the multi-task model
cnn_backbone = CNNBackbone(pretrained=True)
transformer_encoder = TransformerEncoder(input_dim=2048)
model = MultiTaskModel(cnn_backbone, transformer_encoder).to(device)

# AdamW Optimizer and Cosine Annealing Scheduler with updated GradScaler
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = CosineAnnealingLR(optimizer, T_max=10)
scaler = GradScaler('cuda')

from sklearn.metrics import classification_report

## Training Function
def train_model(model, train_loader, val_loader, optimizer, scheduler, num_epochs=5):
    for epoch in range(START_EPOCH, num_epochs):
        model.train()
        running_loss = 0.0
        for batch_idx, (images, class_labels, seg_masks) in enumerate(train_loader):
            images = images.to(device)
            class_labels = class_labels.to(device)
            seg_masks = seg_masks.to(device).float()

            optimizer.zero_grad()
            with autocast():
                class_out, seg_out = model(images)

                # Debug print statements
                print("Segmentation out:", seg_out.shape)
                print("Segmentation mask:", seg_masks.shape)

                seg_out_upsampled = torch.nn.functional.interpolate(
                    seg_out, size=seg_masks.shape[2:], mode='bilinear', align_corners=False
                )
                loss = multi_task_loss(class_out, class_labels, seg_out_upsampled, seg_masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        scheduler.step()

        avg_train_loss = running_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {avg_train_loss:.4f}")

        # Validation
        model.eval()
        val_class_preds, val_class_labels = [], []
        val_seg_preds, val_seg_gts, case_ids = [], [], []

        with torch.no_grad():
            for batch_idx, (images, class_labels, seg_masks) in enumerate(val_loader):
                images = images.to(device)
                class_labels = class_labels.to(device)
                seg_masks = seg_masks.to(device).float()

                class_out, seg_out = model(images)

                seg_out_upsampled = torch.nn.functional.interpolate(
                    seg_out, size=seg_masks.shape[2:], mode='bilinear', align_corners=False
                )

                preds = torch.argmax(class_out, dim=1)
                val_class_preds.extend(preds.cpu().numpy())
                val_class_labels.extend(class_labels.cpu().numpy())

                val_seg_preds.extend(seg_out_upsampled.cpu())
                val_seg_gts.extend(seg_masks.cpu())
                case_ids.extend([f"case_{batch_idx}_{i}" for i in range(images.size(0))])

        # Classification report
        report = classification_report(val_class_labels, val_class_preds, digits=4)
        print(f"Validation Classification Report:\n{report}")

        # Visualize segmentation (optional)
        visualize_predictions(val_seg_preds, val_seg_gts, case_ids)

        # Save model checkpoint
        if (epoch + 1) % SAVE_EVERY == 0:
            save_path = os.path.join(model_dir, f"multitask_model_epoch_{epoch+1}.pth")
            save_checkpoint(model, optimizer, epoch + 1, save_path)
            print(f"Model saved to {save_path}")

# Call the training function
train_model(model, train_loader, val_loader, optimizer, scheduler, num_epochs=EPOCHS)
